In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [6]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 3.97


### Load Player Data and Bookmaker Data

In [14]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

# dfsData.head()

C:\Users\alexg\AppData\Local\Temp\ipykernel_78548\3134731689.py:5: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


### Update projected starting lineups

In [8]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated c:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\PRODUCTION/teamInfo.py
Updated 12 teams with confirmed lineups


### Top EVs for single bets

In [15]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['BOOKMAKER'] != 'Bovada') & (usData['BOOKMAKER'] != 'BetOnline.ag')]

results = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
edge_threshold=0.30, stake=10, variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, 
max_kelly=0.25)  


singleBets = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'EXPECTED ROI', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets with single model...
Pre-computing predictions for 90 unique players...
Error getting prediction for Zach Edey: float division by zero


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,EXPECTED ROI,KELLY_FRACTION,SIGMA FLAG
0,Lauri Markkanen,BetRivers,27.5,34.77,Over,108,1,6.73,67.3,0.623,High
1,Lauri Markkanen,FanDuel,26.5,34.77,Over,102,1,6.73,67.3,0.660,High
2,Lauri Markkanen,BetRivers,26.5,34.77,Over,-107,1,6.28,62.8,0.672,High
3,Keyonte George,BetRivers,20.5,25.05,Over,123,1,6.15,61.5,0.500,High
4,Lauri Markkanen,BetMGM,26.5,34.77,Over,-110,1,5.96,59.6,0.656,High


## Top EVs for 2 leg bets

### Underdog picks

In [10]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=3)

underdogPairs = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 59 players...
Error getting prediction for Zach Edey: float division by zero
Processing 50 players with valid predictions...
Generated 1125 valid 2-leg combinations
Applied player frequency limit (3 max appearances per player)
Selected 75 combinations from 1125 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Lauri Markkanen,Nick Richards,26.5,4.5,34.77,7.39,over,over,0,7.51,0.376,High,Med
1,Lauri Markkanen,Dillon Brooks,26.5,18.5,34.77,23.84,over,over,1,7.30,0.365,High,High
2,Anthony Black,Lauri Markkanen,9.5,26.5,14.28,34.77,over,over,1,7.27,0.364,High,High
3,Anthony Black,Keyonte George,9.5,18.5,14.28,25.05,over,over,1,6.55,0.327,High,High
4,Keyonte George,Nick Richards,18.5,4.5,25.05,7.39,over,over,0,6.52,0.326,High,Med
5,Keyonte George,Dillon Brooks,18.5,18.5,25.05,23.84,over,over,1,6.45,0.322,High,High
6,Draymond Green,Anthony Black,7.5,9.5,11.05,14.28,over,over,0,5.80,0.290,Med,High
7,Draymond Green,Nick Richards,7.5,4.5,11.05,7.39,over,over,0,5.70,0.285,Med,Med
8,Draymond Green,Dillon Brooks,7.5,18.5,11.05,23.84,over,over,0,5.68,0.284,Med,High
9,Tristan da Silva,Day'Ron Sharpe,11.5,5.5,16.21,8.40,over,over,0,4.81,0.240,High,Med


### Prizepicks picks

In [11]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=3)

pairsPrizepicks = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

Pre-computing predictions for 92 players...
Error getting prediction for Zach Edey: float division by zero
Processing 81 players with valid predictions...
Generated 3036 valid 2-leg combinations
Applied player frequency limit (3 max appearances per player)
Selected 119 combinations from 3036 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Lauri Markkanen,Marcus Smart,26.0,7.0,34.77,13.65,over,over,1,9.47,0.474,High,High
1,Lauri Markkanen,Nick Richards,26.0,4.0,34.77,7.39,over,over,0,8.64,0.432,High,Med
2,Marcus Smart,Nick Richards,7.0,4.0,13.65,7.39,over,over,0,8.49,0.425,High,Med
3,Keyonte George,Marcus Smart,18.5,7.0,25.05,13.65,over,over,1,8.07,0.403,High,High
4,Lauri Markkanen,Dillon Brooks,26.0,18.5,34.77,23.84,over,over,1,7.62,0.381,High,High


## 3 leg parlay

### Underdog picks

In [12]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

underdogTrios = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 59 players...
Error getting prediction for Zach Edey: float division by zero
Processing 50 players with valid predictions...
Generated 19118 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 32 combinations from 19118 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Lauri Markkanen,Keyonte George,Nick Richards,26.5,18.5,4.5,34.77,25.05,7.39,over,over,over,0,14.85,0.297,High,High,Med
1,Lauri Markkanen,Keyonte George,Dillon Brooks,26.5,18.5,18.5,34.77,25.05,23.84,over,over,over,1,14.80,0.296,High,High,High
2,Anthony Black,Dillon Brooks,Nick Richards,9.5,18.5,4.5,14.28,23.84,7.39,over,over,over,0,12.06,0.241,High,High,Med
3,Draymond Green,Anthony Black,Day'Ron Sharpe,7.5,9.5,5.5,11.05,14.28,8.40,over,over,over,0,11.01,0.220,Med,High,Med
4,Draymond Green,Tristan da Silva,Day'Ron Sharpe,7.5,11.5,5.5,11.05,16.21,8.40,over,over,over,0,10.35,0.207,Med,High,Med


### Prizepicks picks

In [13]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

triosPrizepicks = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 92 players...
Error getting prediction for Zach Edey: float division by zero
Processing 81 players with valid predictions...
Generated 83831 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 53 combinations from 83831 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Lauri Markkanen,Marcus Smart,Nick Richards,26.0,7.0,4.0,34.77,13.65,7.39,over,over,over,0,18.35,0.367,High,High,Med
1,Lauri Markkanen,Marcus Smart,Dillon Brooks,26.0,7.0,18.5,34.77,13.65,23.84,over,over,over,1,17.07,0.341,High,High,High
2,Keyonte George,Isaiah Collier,Nick Richards,18.5,8.5,4.0,25.05,12.69,7.39,over,over,over,0,14.01,0.280,High,Med,Med
3,Keyonte George,Isaiah Collier,Dillon Brooks,18.5,8.5,18.5,25.05,12.69,23.84,over,over,over,0,12.92,0.258,High,Med,High
4,Anthony Black,Day'Ron Sharpe,Jake LaRavia,9.5,5.5,7.0,14.28,8.40,10.95,over,over,over,0,11.21,0.224,High,Med,High
